# Распознавание животных на изображениях

Базовый ноутбук для домашнего задания по распознаванию животных. 

Чтобы запускать и редактировать код, сохраните копию этого ноутбука себе (Файл -> Создать копию на Диске). Свою копию вы сможете изменять и запускать.

Учебный курс "[Программирование глубоких нейронных сетей на Python](https://openedu.ru/course/urfu/PYDNN/)".

<a target="_blank" href="https://colab.research.google.com/github/sozykin/dlpython_course/blob/master/02_cnn/tensorflow/animals.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Rescaling
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing import image
from tensorflow import data
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
%matplotlib inline

In [ ]:
# Порядок классов животных
class_names=['cat',
             'elephant',
             'butterfly', 
             'sheep',
             'spider',
             'horse',
             'dog',
             'cow',
             'chicken',
             'squirrel']

## Подготовка данных

Загружаем данные для обучения

In [ ]:
!curl -s -L -o train.zip "https://www.dropbox.com/scl/fi/t91ykrvtwupimxlf5xv9f/train.zip?rlkey=k0of0w0x6gsm0l33eylflub9c&dl=1"

Распаковываем архив с обучающим набором данных

In [ ]:
data_path = 'data'

In [ ]:
path = Path(data_path)

if not path.exists():
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
with zipfile.ZipFile('train.zip', 'r') as zip_ref:
    zip_ref.extractall(data_path)

**Создаем DataSet'ы**

In [ ]:
batch_size = 64
image_size = (100, 100)

Набор данных для обучения

In [ ]:
train_dataset = image_dataset_from_directory(
    "data/train", 
    image_size=image_size, 
    batch_size=batch_size,
    validation_split=0.2,
    subset="training",
    class_names=class_names,    
    seed=42
)

Проверочный набор данных

In [ ]:
val_dataset = image_dataset_from_directory(
    "data/train", 
    image_size=image_size, 
    batch_size=batch_size,
    validation_split=0.2,
    subset="validation",
    class_names=class_names,     
    seed=42
)

**Просмотр примеров данных**

In [ ]:
train_dataset.class_names

In [ ]:
for images, labels in train_dataset.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

**Настраиваем производительность работы DataSet'ов**

In [ ]:
AUTOTUNE = data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.prefetch(buffer_size=AUTOTUNE)

## Создаем нейронную сеть

In [ ]:
# Создаем последовательную модель
model = Sequential(
    [
        # Предварительная обработка: нормализация
        Rescaling(1./255),
        # Первый сверточный слой
        Conv2D(32, (3, 3), activation='relu'),
        # Первый слой подвыборки
        MaxPooling2D(pool_size=(2, 2)),
        # Слой регуляризации Dropout
        Dropout(0.25),

        # Второй сверточный слой
        Conv2D(64, (3, 3), activation='relu'),
        # Второй слой подвыборки
        MaxPooling2D(pool_size=(2, 2)),
        # Слой регуляризации Dropout
        Dropout(0.25),

        # Слой преобразования данных из 2D представления в плоское
        Flatten(),
        # Полносвязный слой для классификации
        Dense(512, activation='relu'),
        # Слой регуляризации Dropout
        Dropout(0.5),
        # Выходной полносвязный слой
        Dense(10, activation='softmax')
    ]
)

**Компилируем модель**

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

## Обучаем нейронную сеть

In [ ]:
history = model.fit(train_dataset,
                    epochs=15,
                    validation_data=val_dataset,
                    verbose=1)

In [ ]:
plt.plot(history.history['accuracy'],
         label='Доля верных ответов на обучающем наборе')
plt.plot(history.history['val_accuracy'],
         label='Доля верных ответов на проверочном наборе')
plt.xlabel('Эпоха обучения')
plt.ylabel('Доля верных ответов')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'],
         label='Ошибка на обучающем наборе')
plt.plot(history.history['val_loss'],
         label='Ошибка на проверочном наборе')
plt.xlabel('Эпоха обучения')
plt.ylabel('Ошибка')
plt.legend()
plt.show()

## Сохраняем обученную нейронную сеть

In [ ]:
model.save("animals.keras")

## Применяем сеть для распознавания объектов на изображениях

**Смотрим загруженную картинку**

In [ ]:
img_path = '../horse.jpg'
img = image.load_img(img_path, target_size=(100, 100))
plt.imshow(img)
plt.show()

**Преобразуем картинку в массив для распознавания**

In [ ]:
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)

**Запускаем распознавание**

In [ ]:
prediction = model.predict(x)

In [ ]:
prediction

In [ ]:
prediction = np.argmax(prediction)


In [ ]:
prediction

In [ ]:
print(class_names[prediction])

**Обязательно попробуйте распознать свои изображения!**

## Распознаем изображения их тестового набора данных

Загружаем данные для распознавания

In [ ]:
!curl -s -L -o test.zip "https://www.dropbox.com/scl/fi/ho3jmmnbpddu183p4nug0/test.zip?rlkey=595fmuikf204gq8gzfjvyr4b5&dl=1"

Распаковываем архив

In [ ]:
with zipfile.ZipFile('test.zip', 'r') as zip_ref:
    zip_ref.extractall(data_path)

Тестовый набор данных

In [ ]:
test_dataset = image_dataset_from_directory(
    "data/test", 
    image_size=image_size, 
    batch_size=batch_size,
    labels=None,
    shuffle=False,
)

In [ ]:
for images in test_dataset.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.axis("off")

In [ ]:
predictions = model.predict(test_dataset)

In [ ]:
predictions[:10]

In [ ]:
predictions = np.argmax(predictions, axis=1)

In [ ]:
predictions[:10]

## Готовим файл с решением

In [ ]:
submission = pd.DataFrame({
    'filepath': test_dataset.file_paths,
    'label': predictions
})

In [ ]:
submission

In [ ]:
submission['id'] = submission['filepath'].apply(lambda x: Path(x).name)

In [ ]:
submission

In [ ]:
submission[['id','label']].to_csv('submission.csv',
                                  index=False)

## Идеи по улучшению качества решения

1. Попробуйте использовать разное количество блоков слой свертки + слой подвыборки.
2. Используйте разное количество сверточных слоев в блоке (1, 2, 3).
3. Используйте разные размеры свертки (3х3, 5х5, 7х7).
4. Используйте Data Augmentation.
5. Используйте разное количество нейронов в полносвязном слое.
6. Используйте несколько полносвязных слоев.
7. Используйте разное значение параметра dropout rate.
8. Используйте разное количество эпох: 10, 20, 30, 50, 100.
9. Используйте разные размеры мини-выборки (batch_size): 10, 50, 100, 200, 500.
10. Используйте разные [оптимизаторы](https://keras.io/api/optimizers/): adam, rmsprop и другие.
 

Подберить разные комбинации гиперпараметров таким образом, чтобы получить лучший результат на тестовом наборе данных.

Убедитесь, что в вашей модели нет переобучения.